# Knowledge Graph

_Implementing an agent to query a knowledge graph._

An agent queries a knowledge graph over a graph database using MCP server provided and custom tools.

**Prerequisites:**

Following the instructions below.

1. Open a new **Anaconda Prompt** terminal and activate environment `agentic_ai` using command `conda activate agentic_ai`.

2. Refer to the package installtion section in GitHub README.md and install Neo4j MCP Server in the same environment.

3. Check the installation by running command `neo4j-mcp-server -v` in the same terminal and check if its version is `1.5.3` or later.

4. In the same terminal, run the following command to start Neo4j MCP server and connect to demo database `companies` running in a remote instance. (For the list of other available demo databases, refer to https://neo4j.com/docs/getting-started/appendix/example-data/)

```
neo4j-mcp-server --neo4j-uri "neo4j+s://demo.neo4jlabs.com:7687" --neo4j-username "companies" --neo4j-password "companies" --neo4j-database "companies" --neo4j-read-only true
```

5. Open a new browser window and open the URL `https://demo.neo4jlabs.com:7473/browser/` to browse the demo database. (Note that the interface loads slow. So, don't close the browser window). Once loaded, enter the following details in the connection popup window to connect to the intended database.
```
Protocol: neo4j+s://
Connection URL: demo.neo4jlabs.com:7687
Database user: companies
Password: companies
```

If prompted for default password change, ignore it for this experiment by closing the popup window.

Ensure the database `companies` is selected from dropdown `Database`.



In [ ]:
# Imports packages

from langchain_neo4j import Neo4jGraph
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_ollama import ChatOllama
import json

## Tools

### Custom Tools
_Creating additional custom tools for executing specific Cypher queries directly on the database bypassing assistance from MCP server._

In [2]:
# Creates a connection with the remote demo Neo4j graph instance
neo4j_graph = Neo4jGraph(
    url="neo4j+s://demo.neo4jlabs.com:7687",
    username="companies",
    password="companies",
    database="companies",
    refresh_schema=False    # Flags not to refresh schema information at initialization
    )

In [3]:
@tool
async def get_investments(company: str) -> str:
    """
    Returns the investments by a company by name. Returns list of investment ids, names and types.
    
    Parameters:
    - company (str): The name of the company to fetch investments for.

    Returns:
    - str: A JSON-formatted string containing a list of investments, each with its id, name, and type.
    """
    
    try:
        results = neo4j_graph.query("""
            MATCH (o:Organization)-[:HAS_INVESTOR]->(i)
            WHERE o.name = $company
            RETURN i.id as id, i.name as name, head(labels(i)) as type
        """, {"company": company})
        return json.dumps(results, indent=2)
    
    except Exception as e:
        raise Exception(f"Error fetching investments: {str(e)}")

In [ ]:
# Checks if the custom tool works as expected by invoking it with a sample company name.
await get_investments.ainvoke("Neo4j")

'[\n  {\n    "id": "Rod Johnson",\n    "name": "Rod Johnson",\n    "type": "Person"\n  }\n]'

### MCP Tools

In [5]:
# MCP client that connects to the (already running) local Neo4j MCP server.

client = MultiServerMCPClient({
    "neo4j": {
        "command": "python",
        "args": ["-m", "neo4j_mcp_server"],
        "env": {
            "NEO4J_URI": "neo4j+s://demo.neo4jlabs.com:7687",
            "NEO4J_USERNAME": "companies",
            "NEO4J_PASSWORD": "companies",
            "NEO4J_DATABASE": "companies",
            "NEO4J_READ_ONLY": "true",
        },
        "transport": "stdio"
    }     
})

In [ ]:
mcp_tools = await client.get_tools()            # Gets list of (LangChain) tools from the Neo4j MCP server.

custom_tools = mcp_tools + [get_investments]    # Combines custom tool with MCP tools into a single list

custom_tools    # Lists all the available tools, including the custom one and those fetched from the Neo4j MCP server.

[StructuredTool(name='get-schema', description='\n\t\tRetrieve the schema information from the Neo4j database, including node labels, relationship types, and property keys.\n\t\tIf the database contains no data, no schema information is returned.', args_schema={'properties': {}, 'required': [], 'type': 'object'}, metadata={'title': 'Get Neo4j Schema', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': True}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c167453e5c0>),
 StructuredTool(name='list-gds-procedures', description='Use this tool to discover what graph science and analytics functions are available in the current Neo4j environment. It returns a structured list describing each function — what it does, how to use it, the inputs it needs, and what kind of results it produces. Do this before any reasoning, query generation, or analysis so you know what capabilities exist. 

In [9]:
# Creates a dictionary mapping tool names to their corresponding tool objects for easy lookup during tool call.
tools_by_name = {tool.name: tool for tool in custom_tools}      

tools_by_name   # Displays the above mapping just for reference.

{'get-schema': StructuredTool(name='get-schema', description='\n\t\tRetrieve the schema information from the Neo4j database, including node labels, relationship types, and property keys.\n\t\tIf the database contains no data, no schema information is returned.', args_schema={'properties': {}, 'required': [], 'type': 'object'}, metadata={'title': 'Get Neo4j Schema', 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': True}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c167453e5c0>),
 'list-gds-procedures': StructuredTool(name='list-gds-procedures', description='Use this tool to discover what graph science and analytics functions are available in the current Neo4j environment. It returns a structured list describing each function — what it does, how to use it, the inputs it needs, and what kind of results it produces. Do this before any reasoning, query generation, or analysis 

## Model

_Connecting an appropriate model to a chat client._

In [ ]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "granite4.1:3b"  # In this experiment, model Granite 4.1 3B is preferred over 
                                # Llama 3.2 3B and Qwen 3.5 2B/4B as these were observed being failing 
                                # intermitently to prepare a valid Cypher query from graph scheme. 
                                # Refer to model installation section in README.md.

OLLAMA_ENDPOINT = "http://localhost:11434/"

In [ ]:
# Initializes chat client
chat_model = ChatOllama(
    model=OLLAMA_MODEL,
    reasoning=False,
    temperature=0.0,
    base_url=OLLAMA_ENDPOINT)

model_with_tools = chat_model.bind_tools(custom_tools)      # Binds the chat model with the list of tools.


## Agent

In [13]:
# Instruction to be used to set agent's behaviour

system_message = """You are a company research assistant with access to a Neo4j graph database and news data.

Available tools:
- get-schema: Retrieve the database schema. Always call this first before writing Cypher queries. Contains information about companies, people, and similar.
- read-cypher: Execute Cypher queries against the database. Use the schema to construct valid queries. Contains information about companies, people, and similar.
- get_investments: Retrieve investment and funding data.

When answering questions:
1. Always use the available tools to gather information—do not rely on prior knowledge.
2. If the question involves database queries, first call get_schema to understand the data model.
3. Write precise Cypher queries based on the actual schema—don't assume node labels or relationship types.
4. Combine multiple tools when needed for comprehensive answers."""


In [ ]:
async def execute_query(query):
    """
    A helper function to execute a query using the chat model and tools.

    Parameters:
    - query (str): The query to be executed.

    Returns:
    - str: The result of the query execution.
    """

    print(f"===Human Message===\n{query}")
    
    messages = [
        SystemMessage(content=system_message)               # Sets the system message to define the agent's behavior and available tools.
    ]
    messages.append(HumanMessage(query))                    # Appends the human message (the query) to the messages list.
    print("\n===Messages to Model==="); display(messages)
    
    ai_message = model_with_tools.invoke(messages)          # Invokes the model with the current messages to get the AI's response.
    messages.append(ai_message)
    print("\n===AI Message==="); display(ai_message)

    # Loops to handle tool calls until the AI message no longer contains any tool calls.
    while ai_message.tool_calls:
        for tool_call in ai_message.tool_calls:
            tool = tools_by_name[tool_call["name"]]                 # Gets the tool object corresponding to the tool call name
            tool_message = await tool.ainvoke(tool_call["args"])    # Invokes the tool with the provided arguments and gets response
            messages.append(ToolMessage(content=tool_message, tool_call_id=tool_call["id"]))    # Adds the tool's response to the messages list
            print("\n===Tool Message==="); display(tool_message)

        print("\n===Messages to Model==="); display(messages)
        ai_message = model_with_tools.invoke(messages)      # Invokes the model again with the updated messages to get the next AI response after tool calls.
        messages.append(ai_message)
        print("\n===AI Message==="); display(ai_message)

    return ai_message.content

## Testing

_Testing agent over MCP and custom tools._

**Testing agent over MCP tools**

In [15]:
user_query = "Who is the CEO of Neo4j?"

final_response = await execute_query(user_query)

print(f"Final Response: {final_response}")

===Human Message===
Who is the CEO of Neo4j?

===Messages to Model===


[SystemMessage(content="You are a company research assistant with access to a Neo4j graph database and news data.\n\nAvailable tools:\n- get-schema: Retrieve the database schema. Always call this first before writing Cypher queries. Contains information about companies, people, and similar.\n- read-cypher: Execute Cypher queries against the database. Use the schema to construct valid queries. Contains information about companies, people, and similar.\n- get_investments: Retrieve investment and funding data.\n\nWhen answering questions:\n1. Always use the available tools to gather information—do not rely on prior knowledge.\n2. If the question involves database queries, first call get_schema to understand the data model.\n3. Write precise Cypher queries based on the actual schema—don't assume node labels or relationship types.\n4. Combine multiple tools when needed for comprehensive answers.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Who is the CEO of Neo4j?',


===AI Message===


AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-06-15T05:30:09.991018454Z', 'done': True, 'done_reason': 'stop', 'total_duration': 34614160116, 'load_duration': 2296716637, 'prompt_eval_count': 781, 'prompt_eval_duration': 30857128905, 'eval_count': 16, 'eval_duration': 1416646618, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019ec9c1-b38d-7ae1-b797-900c73ba11bf-0', tool_calls=[{'name': 'get-schema', 'args': {}, 'id': 'aa088dc7-c682-4029-a6ab-1bdb56a361df', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 781, 'output_tokens': 16, 'total_tokens': 797})


===Tool Message===


...


===Messages to Model===


[SystemMessage(content="You are a company research assistant with access to a Neo4j graph database and news data.\n\nAvailable tools:\n- get-schema: Retrieve the database schema. Always call this first before writing Cypher queries. Contains information about companies, people, and similar.\n- read-cypher: Execute Cypher queries against the database. Use the schema to construct valid queries. Contains information about companies, people, and similar.\n- get_investments: Retrieve investment and funding data.\n\nWhen answering questions:\n1. Always use the available tools to gather information—do not rely on prior knowledge.\n2. If the question involves database queries, first call get_schema to understand the data model.\n3. Write precise Cypher queries based on the actual schema—don't assume node labels or relationship types.\n4. Combine multiple tools when needed for comprehensive answers.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Who is the CEO of Neo4j?',


===AI Message===


AIMessage(content='The CEO of Neo4j is **Emil\u202fEifrem**.', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-06-15T05:31:47.664643972Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7864241004, 'load_duration': 67869716, 'prompt_eval_count': 1777, 'prompt_eval_duration': 4846934240, 'eval_count': 18, 'eval_duration': 2912023864, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019ec9c3-9996-73f3-ab1a-5b40de0bd163-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1777, 'output_tokens': 18, 'total_tokens': 1795})

Final Response: The CEO of Neo4j is **Emil Eifrem**.


**Testing agent over custom tools**

In [16]:
user_query = "Who are the investors for Neo4j?"

final_response = await execute_query(user_query)

print(f"Final Response: {final_response}")

===Human Message===
Who are the investors for Neo4j?

===Messages to Model===


[SystemMessage(content="You are a company research assistant with access to a Neo4j graph database and news data.\n\nAvailable tools:\n- get-schema: Retrieve the database schema. Always call this first before writing Cypher queries. Contains information about companies, people, and similar.\n- read-cypher: Execute Cypher queries against the database. Use the schema to construct valid queries. Contains information about companies, people, and similar.\n- get_investments: Retrieve investment and funding data.\n\nWhen answering questions:\n1. Always use the available tools to gather information—do not rely on prior knowledge.\n2. If the question involves database queries, first call get_schema to understand the data model.\n3. Write precise Cypher queries based on the actual schema—don't assume node labels or relationship types.\n4. Combine multiple tools when needed for comprehensive answers.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Who are the investors for 


===AI Message===


AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-06-15T05:31:52.463063639Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4778000097, 'load_duration': 138210698, 'prompt_eval_count': 781, 'prompt_eval_duration': 1386659428, 'eval_count': 24, 'eval_duration': 3210017468, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019ec9c3-b863-79a1-9d26-2cc9e2195236-0', tool_calls=[{'name': 'get_investments', 'args': {'company': 'Neo4j'}, 'id': 'd3de8ccf-c0ad-4e2e-850d-f9df08a7b168', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 781, 'output_tokens': 24, 'total_tokens': 805})


===Tool Message===


...


===Messages to Model===


[SystemMessage(content="You are a company research assistant with access to a Neo4j graph database and news data.\n\nAvailable tools:\n- get-schema: Retrieve the database schema. Always call this first before writing Cypher queries. Contains information about companies, people, and similar.\n- read-cypher: Execute Cypher queries against the database. Use the schema to construct valid queries. Contains information about companies, people, and similar.\n- get_investments: Retrieve investment and funding data.\n\nWhen answering questions:\n1. Always use the available tools to gather information—do not rely on prior knowledge.\n2. If the question involves database queries, first call get_schema to understand the data model.\n3. Write precise Cypher queries based on the actual schema—don't assume node labels or relationship types.\n4. Combine multiple tools when needed for comprehensive answers.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Who are the investors for 


===AI Message===


AIMessage(content='Neo4j’s founding investors include **Rod Johnson**, a well‑known early investor in the company. (The investment data source only lists Rod\u202fJohnson as an investor for Neo4j.)', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-06-15T05:32:08.02884943Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7035707274, 'load_duration': 68396953, 'prompt_eval_count': 884, 'prompt_eval_duration': 1619235250, 'eval_count': 41, 'eval_duration': 5284449892, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019ec9c3-ec60-7ba1-91fb-6834fe82d7fc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 884, 'output_tokens': 41, 'total_tokens': 925})

Final Response: Neo4j’s founding investors include **Rod Johnson**, a well‑known early investor in the company. (The investment data source only lists Rod Johnson as an investor for Neo4j.)
